# Lab Orpheus — chạy **từng cell theo thứ tự**

**Không bấm Run all** cả notebook.

Orpheus trên Colab cần **một lần Restart** giữa bước cài package và bước import.

## Thứ tự chạy (làm đúng như bảng)

| Bước | Cell | Việc bạn làm |
|------|------|----------------|
| 1 | **Cell 1** | Config URL repo |
| 2 | **Cell 2** | Clone / pull code |
| 3 | **Cell 3** | **Chỉ pip install** (không import Orpheus) |
| 4 | **Menu Colab** | **Runtime → Restart session** |
| 5 | **Cell 1** rồi **Cell 2** | Gán lại biến + `cd` vào repo (sau restart mất biến) |
| 6 | **Cell 4** | Import Orpheus + kiểm GPU — phải thấy `OrpheusModel import OK` |
| 7 | **Cell 5** | `prepare` manifest Stage 1–2 |
| 8 | **Cell 6** | Render smoke 4 đoạn (tara/leo) |
| 9 | **Cell 7** | Nghe tracking DISPLAY / SPOKEN / audio |
| 10 | **Cell 8** | (Tuỳ chọn) lưu review |
| 11 | **Cell 9** | (Tuỳ chọn) ghép 1 file wav |
| 12 | **Cell 10** | (Tuỳ chọn) survey nhiều giọng |

### Nếu lỗi `numpy.dtype size changed`
Bạn đã import ngay sau khi pip trong cùng session.  
→ **Restart session** (hoặc Delete runtime) rồi làm lại từ **bước 5–6** (Cell 1, 2, 4). Không cần cài lại trừ khi import vẫn fail.

### Docs
- Upstream: https://github.com/canopyai/Orpheus-TTS (`orpheus-speech` + `vllm==0.7.3`)
- Colab mẫu Orpheus: https://colab.research.google.com/drive/1KhXT56UePPUHhqitJNUxq63k-pQomz3N


## Cell 1 — Config


In [ ]:
# CELL 1 - Config
REPO_URL = "https://github.com/phamtu2x5/ielts-script2audio.git"
BRANCH = "main"
WORKDIR = "/content/ielts-script2audio"
print("OK Cell 1")


## Cell 2 — Clone / pull (sau Cell 1)


In [ ]:
# CELL 2 - Clone or pull
import os
from pathlib import Path

if not Path(WORKDIR).exists():
    !git clone --branch {BRANCH} {REPO_URL} {WORKDIR}
else:
    %cd {WORKDIR}
    !git pull origin {BRANCH}
%cd {WORKDIR}
assert Path("pyproject.toml").is_file(), "Chay lai Cell 1 roi Cell 2"
print("OK Cell 2 CWD=", os.getcwd())


## Cell 3 — CHỈ CÀI PACKAGE (không import Orpheus)

Chạy sau Cell 2. Cell này **chỉ pip**.

### Khi pip xong:
1. Menu: **Runtime → Restart session**
2. Chạy lại **Cell 1** và **Cell 2**
3. Chạy **Cell 4** (import)
4. **Khong** can chay lai Cell 3 neu pip da OK


In [ ]:
# CELL 3 - INSTALL ONLY (do not import vllm / OrpheusModel here)
import sys
import subprocess

def pipi(*args):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *args]
    print(">", " ".join(cmd), flush=True)
    subprocess.check_call(cmd)

print("python", sys.version)
pipi("-e", ".[dev]")
subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "orpheus-tts"])

# Upstream: https://github.com/canopyai/Orpheus-TTS
pipi("orpheus-speech")
pipi("vllm==0.7.3")
pipi("--force-reinstall", "numpy==1.26.4")
pipi("--force-reinstall", "pillow==10.4.0")
pipi("soundfile")

print("OK Cell 3 pip xong.")
print(">>> BAT BUOC: Runtime -> Restart session")
print(">>> Sau do: Cell 1 -> Cell 2 -> Cell 4")


## ⏸ RESTART SESSION TẠI ĐÂY

**Runtime → Restart session**

Sau restart chạy: **Cell 1 → Cell 2 → Cell 4** (bỏ qua Cell 3 nếu cài đã xong).


## Cell 4 — Import + GPU (sau Restart)


In [ ]:
# CELL 4 - IMPORT CHECK (after Restart only)
import sys
print("python", sys.version)

import numpy as np
print("numpy", np.__version__)

import torch
print("torch", torch.__version__, "cuda", torch.version.cuda)
print("cuda_available", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Bat GPU: Runtime -> Change runtime type -> GPU")
print("device", torch.cuda.get_device_name(0))
print("vram_gb", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

import vllm
from orpheus_tts import OrpheusModel
print("vllm", getattr(vllm, "__version__", "?"))
print("OK Cell 4: OrpheusModel import OK -> tiep Cell 5")


## Cell 5 — prepare manifest (Stage 1–2)


In [ ]:
# CELL 5 - prepare
import json
from pathlib import Path

Path("outputs").mkdir(exist_ok=True)
Path("lab_audio/orpheus_part1").mkdir(parents=True, exist_ok=True)
!ielts-s2a prepare examples/part1_valid.json --pretty -o outputs/part1_manifest.json
m = json.loads(Path("outputs/part1_manifest.json").read_text(encoding="utf-8"))
assert m["validation"]["valid"] is True
print("OK Cell 5 n_requests=", len(m["requests"]))
for u in m["prepared_utterances"]:
    if u["event_id"] in {"e004", "e006", "e008", "e011"}:
        print(u["event_id"], u["spoken_text"][:80])


## Cell 6 — Render smoke 4 đoạn (tara/leo). Xoá `--limit 4` nếu muốn full.


In [ ]:
# CELL 6 - render smoke
import json
import os
from pathlib import Path

from orpheus_tts import OrpheusModel
print("preflight OrpheusModel OK")

OUT = Path("lab_audio/orpheus_part1")
REPORT = OUT / "lab_render_report.json"
OUT.mkdir(parents=True, exist_ok=True)

cmd = (
    "python scripts/lab_render_orpheus_from_manifest.py "
    "--manifest outputs/part1_manifest.json "
    "--voice-map configs/voice_maps/orpheus_en_part1.json "
    "--out-dir lab_audio/orpheus_part1 "
    "--temperature 0.6 --repetition-penalty 1.3 --limit 4"
)
print(cmd)
ret = os.system(cmd)
print("exit", ret)
if not REPORT.is_file():
    raise FileNotFoundError("Khong co report - Cell 4 import fail hoac render fail")
report = json.loads(REPORT.read_text(encoding="utf-8"))
print("OK Cell 6 ok=", report["ok_count"], "fail=", report["fail_count"])
for s in report["segments"]:
    print(s["segment_id"], s["status"], s.get("backend_voice_id"), s.get("error") or "")


## Cell 7 — Tracking: DISPLAY + SPOKEN + audio


In [ ]:
# CELL 7 - track
from pathlib import Path
from IPython.display import Audio, display
import json

REPORT_PATH = Path("lab_audio/orpheus_part1/lab_render_report.json")
assert REPORT_PATH.is_file(), "Chay Cell 6 truoc"
report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
segments = report.get("segments") or []
audio_dir = REPORT_PATH.parent
print(report.get("transcript_id"), "n=", len(segments))
for i, seg in enumerate(segments, start=1):
    fname = seg.get("output_filename")
    wav = audio_dir / fname if fname else None
    print("=" * 72)
    print(f"[{i}/{len(segments)}] {seg.get('segment_id')} | {seg.get('speaker_id')} | {seg.get('backend_voice_id')}")
    print("DISPLAY:", seg.get("display_text", ""))
    print("SPOKEN :", seg.get("spoken_text", ""))
    if wav is not None and wav.is_file():
        display(Audio(filename=str(wav)))
    else:
        print("missing", fname)
print("OK Cell 7")


## Cell 8 — Lưu review (tuỳ chọn)


In [ ]:
# CELL 8 - reviews
from pathlib import Path
import json

report = json.loads(Path("lab_audio/orpheus_part1/lab_render_report.json").read_text(encoding="utf-8"))
reviews = {s["segment_id"]: {"content_match": "", "notes": ""} for s in report.get("segments") or []}
# reviews["seg_0001"]["content_match"] = "yes"

filled = []
missing = []
for s in report.get("segments") or []:
    sid = s["segment_id"]
    h = reviews.get(sid) or {}
    match = (h.get("content_match") or "").strip().lower()
    if match not in {"yes", "partial", "no"}:
        missing.append(sid)
    filled.append({
        "segment_id": sid,
        "display_text": s.get("display_text"),
        "spoken_text": s.get("spoken_text"),
        "backend_voice_id": s.get("backend_voice_id"),
        "output_filename": s.get("output_filename"),
        "content_match": match,
        "notes": h.get("notes", ""),
    })
out = Path("lab_audio/orpheus_part1/segment_review_filled.json")
out.write_text(json.dumps(filled, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("OK Cell 8", out, "missing", missing)


## Cell 9 — Ghép 1 file (tuỳ chọn)


In [ ]:
# CELL 9 - concat
import json
from pathlib import Path
from IPython.display import Audio, display

assert Path("lab_audio/orpheus_part1/lab_render_report.json").is_file()
!python scripts/lab_concat_segment_wavs.py --report lab_audio/orpheus_part1/lab_render_report.json --out lab_audio/orpheus_part1/part1_full.wav --gap-mode adaptive --same-speaker-gap-ms 220 --turn-gap-ms 520
full = Path("lab_audio/orpheus_part1/part1_full.wav")
print(json.loads(Path("lab_audio/orpheus_part1/part1_full.concat.json").read_text()).get("duration_sec"), "sec")
if full.is_file():
    display(Audio(filename=str(full)))
print("OK Cell 9")


## Cell 10 — Survey nhiều giọng (tuỳ chọn, lâu)


In [ ]:
# CELL 10 - survey
from pathlib import Path
from IPython.display import Audio, display
import json
from collections import defaultdict

from orpheus_tts import OrpheusModel
print("preflight OK")

!python scripts/lab_survey_orpheus_voices.py --manifest outputs/part1_manifest.json --inventory configs/voice_maps/orpheus_voice_inventory.json --preset en_shortlist --out-dir lab_audio/orpheus_voice_survey --event-ids e004,e006,e008,e011 --end-pad-ms 450

rp = Path("lab_audio/orpheus_voice_survey/voice_survey_report.json")
assert rp.is_file(), "Survey fail"
report = json.loads(rp.read_text(encoding="utf-8"))
print("OK Cell 10", report.get("ok_count"), report.get("fail_count"), report.get("voices"))
by = defaultdict(list)
for r in report.get("renders") or []:
    by[r.get("event_id")].append(r)
for eid, items in by.items():
    items = sorted(items, key=lambda x: x.get("voice_id") or "")
    print("=" * 72, "LINE", eid)
    if items:
        print("DISPLAY", items[0].get("display_text"))
        print("SPOKEN ", items[0].get("spoken_text"))
    for it in items:
        print("---", it.get("voice_id"), it.get("status"))
        p = Path("lab_audio/orpheus_voice_survey") / it["output_filename"]
        if p.is_file():
            display(Audio(filename=str(p)))


Xong. Verdict lab only — `not_selected`. Không sửa script.
